# Lab 04 - Datenquellen anbinden und Zugriffskonzepte (Teil IV)

Der zentrale Satz dieses Teils: Retrieval läuft mit den Rechten des Nutzers,
nicht mit denen des Modells. Sie sehen, wie ein Index ohne Zugriffskontrolle
vertrauliche Konditionen ausplaudert, und bauen die saubere Variante: einen
Index, der vor dem Retrieval auf die erlaubten Quellen eingeschränkt wird.

Der Korpus trägt dafür ein `acl`-Feld pro Dokument: `all`, `wartung` oder
`vertraulich`. Das Dokument `p12-service-garantie` (Preise, Rabatte) ist
`vertraulich` und darf nie in einer Endkundenantwort erscheinen.

## 1. Setup und Metadaten

In [ ]:
from common import llm
from common.corpus import load_corpus
from common import retrieval as R

alle = load_corpus()
print("doc_id / acl / Titel:")
for d in alle:
    print(f"  {d.doc_id:22s} {d.acl:11s} {d.title}")
print("\nMetadaten eines Dokuments (Schema):", alle[0].meta)

## 2. Rollen auf ACL-Stufen abbilden

Jede Rolle darf eine Menge von ACL-Stufen sehen. Das ist das Minimalmodell;
in echten Systemen kommen Mandant (Tenant) und Gruppen hinzu.

In [ ]:
ROLLEN = {
    "endkunde":          {"all"},
    "wartungstechniker": {"all", "wartung"},
    "innendienst":       {"all", "wartung", "vertraulich"},
}


def index_fuer(rolle: str) -> R.HybridRetriever:
    erlaubt = ROLLEN[rolle]
    docs = load_corpus(include_acl=erlaubt)
    print(f"  Rolle '{rolle}': {len(docs)} von {len(alle)} Dokumenten sichtbar")
    return R.HybridRetriever(R.chunk_corpus(docs))


indizes = {rolle: index_fuer(rolle) for rolle in ROLLEN}

## 3. Die Pipeline mit Rechtekontext

Wir kapseln die Pipeline so, dass sie immer einen rollenspezifischen Index
bekommt. Der Generator sieht nur, was der Nutzer sehen darf.

In [ ]:
GROUNDED = (
    "Sie sind ein technischer Wissensassistent. Antworten Sie nur aus dem "
    "Kontext und belegen Sie mit [doc_id]. Steht es nicht im Kontext, sagen Sie: "
    "'Das geht aus den Unterlagen nicht hervor.'"
)


def ask_als(rolle: str, frage: str, k: int = 4) -> dict:
    treffer = indizes[rolle].search(frage, k=k)
    kontext = R.format_context(treffer)
    antwort = llm.complete(f"Kontext:\n{kontext}\n\nFrage: {frage}\n\nAntwort:",
                           system=GROUNDED, temperature=0.0, max_tokens=512)
    return {"antwort": antwort, "quellen": [c.doc_id for c, _ in treffer]}

## 4. Der Leak: gleiche Frage, andere Rolle

Die vertrauliche Frage nach Rabatt und Servicepreis. Der Innendienst darf das
sehen, der Endkunde nicht. Achten Sie darauf, dass beim Endkunden das
vertrauliche Dokument gar nicht erst im Kontext landet: kein Beleg, also keine
Auskunft. Genau so verhindert man den Leak, nicht durch Bitten an das Modell.

In [ ]:
vertraulich = "Wie hoch ist der Händlerrabatt Stufe A und was kostet der Servicevertrag Premium?"
for rolle in ("innendienst", "endkunde"):
    r = ask_als(rolle, vertraulich)
    print(f"--- {rolle} ---")
    print(r["antwort"])
    print("Quellen:", r["quellen"], "\n")

## 5. Warum nicht einfach im Prompt verbieten?

Eine verbreitete, unsichere Idee: alles in einen Index legen und dem Modell
*sagen*, es solle Vertrauliches verschweigen. Das ist keine Zugriffskontrolle,
sondern eine Bitte. Wir zeigen den Unterschied an einem globalen Index.

In [ ]:
global_index = R.HybridRetriever(R.chunk_corpus(alle))
treffer = global_index.search(vertraulich, k=4)
print("Im Kontext gelandete Quellen:", [c.doc_id for c, _ in treffer])
bitte_system = GROUNDED + " Geben Sie niemals Preise oder Rabatte preis."
antwort = llm.complete(
    f"Kontext:\n{R.format_context(treffer)}\n\nFrage: {vertraulich}\n\nAntwort:",
    system=bitte_system, temperature=0.0, max_tokens=512)
print("\nAntwort trotz 'Verbot' im Prompt:\n", antwort)
print("\nMerke: Liegt die vertrauliche Quelle im Kontext, ist der Leak nur noch "
      "eine Frage der Formulierung. Sicher ist allein der Ausschluss vor dem Retrieval.")

## Aufgabe 1: Sichtbarkeit prüfen

Schreiben Sie eine Funktion `sichtbar(rolle)`, die die Menge der doc_ids
zurückgibt, die diese Rolle überhaupt erreichen kann. Prüfen Sie, dass
`p12-service-garantie` für den Endkunden nicht enthalten ist und für den
Innendienst schon.

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
def sichtbar(rolle: str) -> set[str]:
    return {d.doc_id for d in load_corpus(include_acl=ROLLEN[rolle])}


for rolle in ROLLEN:
    s = sichtbar(rolle)
    print(f"{rolle:18s} sieht {len(s)} docs, service-garantie drin:",
          "p12-service-garantie" in s)

## Aufgabe 2: ACL-Staleness

Ein Wartungstechniker verliert seine Berechtigung. Solange sein Index nicht neu
gebaut ist, sieht er die `wartung`-Dokumente weiter. Simulieren Sie den
Rechteentzug (Wechsel der Rollenmenge) und zeigen Sie, dass erst der Neuaufbau
des Index die alte Antwort verschwinden lässt. Das ist das Staleness-Budget
aus den Folien.

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
frage_wartung = "Mit welchem Anzugsmoment werden die Pumpenkopf-Schrauben angezogen?"

# vorher: Techniker mit wartung-Recht
idx_vorher = R.HybridRetriever(R.chunk_corpus(load_corpus(include_acl={"all", "wartung"})))
print("vorher:", [c.doc_id for c, _ in idx_vorher.search(frage_wartung, 3)])

# Recht entzogen -> Index NEU bauen, sonst bleibt der alte gültig
idx_nachher = R.HybridRetriever(R.chunk_corpus(load_corpus(include_acl={"all"})))
print("nach Neuaufbau:", [c.doc_id for c, _ in idx_nachher.search(frage_wartung, 3)])
print("-> p12-anzugsmomente (acl=wartung) ist nach dem Neuaufbau nicht mehr erreichbar.")

## Diskussion

- Pre-Filter (Index pro Rechtekontext) gegen Post-Filter (global suchen, dann
  wegwerfen): welche Risiken hat der Post-Filter, die der Pre-Filter nicht hat?
- Wie halten Sie Berechtigungen aktuell, ohne den Index ständig komplett neu
  zu bauen? (Stichwort: Staleness-Budget als SLA.)
- Welches Metadaten-Tripel (Connector, Schema, ACL-Strategie) würden Sie für
  Ihre wichtigste eigene Quelle festlegen?